# Rural Carbon Risk Classification Model (Model 2)

This notebook implements the training pipeline for **Module 2**: Predicting `Emission_Risk` levels (Low, Medium, High) based on rural farm agricultural and structural footprint. 

The strategy mimics the logic built in `src/model_training/m2_rural_classification.py`. We initialize a robust XGBoost Classifier and map our target definitions via Label Encoding before serializing the trained model to disk.

## 1. Import Dependencies
Include standard Data Science stack (Pandas, Numpy), Scikit-Learn tools (LabelEncoder, Evaluation metrics, Test Train Splitting) alongside the XGBoost Library. Then load settings via the internal `src.config` script.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

# Scikit-Learn Validation & XGBoost 
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Give Jupyter notebook relative access back to src/
sys.path.append(str(Path.cwd().parent.parent))

# Configuration and Paths
from src.config import (
    PROCESSED_FILES,
    WEIGHTS_DIR,
    PREPROCESS_DIR,
    RURAL_RISK_COL,
    RURAL_TARGET_COL,
    MODULE2_PARAMS,
    RANDOM_SEED
)

## 2. Load Preprocessed Features and Target

Grab our dataset encoded with dummy variables during the notebook steps (`08_data_processing_analysis`). Here we will also drop the continuous float carbon target to avoid "data leakage" - `Carbon_Emission_tCO2`. If we don't, mapping `Low`, `Medium`, and `High` becomes a 100% accurate proxy.

In [2]:
data_path = PROCESSED_FILES["rural_carbon"]
print(f"Loading features from {data_path.name}...\n")

try:
    df = pd.read_csv(data_path)
    
    # 2a. Avoid label & target leakage 
    # Drop continuous float column `Carbon_Emission_tCO2` completely 
    cols_to_drop = [RURAL_RISK_COL]
    if RURAL_TARGET_COL in df.columns:
        cols_to_drop.append(RURAL_TARGET_COL)
        print(f"⚠️ Dropped target continuous float column `{RURAL_TARGET_COL}` safely")
        
    X = df.drop(columns=cols_to_drop)
    y_raw = df[RURAL_RISK_COL]

    print(f"\nFeature Tensor (X): {X.shape}")
    display(X.head())

except FileNotFoundError:
    print(f"ERROR: {data_path} not found.")

Loading features from rural_carbon_processed.csv...

⚠️ Dropped target continuous float column `Carbon_Emission_tCO2` safely

Feature Tensor (X): (3000, 314)


,Month,Fertilizer_Usage_kg,Crop_Area_ha,Livestock_Cows,Livestock_Pigs,Household_Energy_kWh,Renewable_Energy_Fraction,Temperature_C,Rainfall_mm,Year,...,Region_R0294,Region_R0295,Region_R0296,Region_R0297,Region_R0298,Region_R0299,Region_R0300,Crop_Type_Maize,Crop_Type_Rice,Crop_Type_Wheat
0,-0.757329,-0.407911,0.739476,-1.703205,0.516532,-0.085013,0.257045,0.381142,-1.095227,1.509256,...,False,False,False,False,False,False,False,True,False,False
1,1.276939,-0.537377,1.517671,1.594777,1.660017,0.898596,-0.017246,0.532377,1.458223,0.011648,...,False,False,False,False,False,False,False,False,False,True
2,0.114500,-1.496811,0.076854,1.178190,0.343277,1.516639,1.594214,0.171740,-1.531608,0.510851,...,False,False,False,False,False,False,False,True,False,False
3,0.405110,-0.028761,0.123083,0.136722,-0.315094,-1.402628,-1.285842,1.207115,1.582904,1.509256,...,False,False,False,False,False,False,False,False,True,False
4,-1.338548,-1.334979,-0.154293,-0.106288,1.192228,-0.634677,1.354209,0.346242,-1.026653,-0.487555,...,False,False,False,False,False,False,False,True,False,False


## 3. Encode the Target and Split

XGBClassifier currently requires numerical array integers for labels like `0, 1, 2` corresponding to `Low, Medium, High`. 
We will split with standard `.2` test size on proportional stratifield classes.

In [3]:
# Fit encoder array back onto [0,1,2] structure
le = LabelEncoder()
y = le.fit_transform(y_raw)

# Show encoding labels map (Low / Med / High => integer representation map)
class_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"Label Encoder Class Map: {class_mapping}\n")

# Split proportionally into Train/Test subsets 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=RANDOM_SEED, 
    stratify=y
)

print(f"X_train -> {X_train.shape}")
print(f"X_test --> {X_test.shape}")

Label Encoder Class Map: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}

X_train -> (2400, 314)
X_test --> (600, 314)


## 4. Train Baseline Classification 

Initialize standard tree building on `eval_metric` `mlogloss`. Passing the array over XGBClassifier natively.

In [4]:
# Fetch global baseline parameters and pass explicitly to prevent deprecated warnings
xgb_params = MODULE2_PARAMS.get("xgb", {})

model = XGBClassifier(
    **xgb_params,
    use_label_encoder=False,
    eval_metric='mlogloss',
    n_jobs=-1
)

print(f"XGBClassifier initialized with:\n{model.get_params()}\n")
print("Training Base classifier... 🌳")

# Execute Base training
model.fit(X_train, y_train)

print("\nFinished ✅")

XGBClassifier initialized with:
{'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': None, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': False, 'eval_metric': 'mlogloss', 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.1, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 6, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 200, 'n_jobs': -1, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': None, 'tree_method': None, 'validate_parameters': None, 'verbosity': None, 'use_label_encoder': False}

Training Base cla

c:\Users\justi\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\training.py:183: UserWarning: [17:02:20] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Finished ✅


## 5. Model Inference & Validation Review

Since this is multi-class, we want an average `accuracy` matrix but more importantly, standard `F1`, `Recall`, and `Precision` parameters mapping out true positives and negatives across High vs Low classifications.

In [5]:
print("--- Evaluating Baseline Model ---\n")
y_pred = model.predict(X_test)

# General accuracy metric across all targets overall 
acc = accuracy_score(y_test, y_pred)
print(f"Overall Base Accuracy: {acc * 100:.2f}%\n")

# Provide mapping report using real text string targets on inference matrix 
print("Classification Scoring Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

--- Evaluating Baseline Model ---

Overall Base Accuracy: 53.50%

Classification Scoring Report:
              precision    recall  f1-score   support

        High       0.58      0.63      0.60       200
         Low       0.62      0.63      0.62       200
      Medium       0.39      0.34      0.37       200

    accuracy                           0.54       600
   macro avg       0.53      0.54      0.53       600
weighted avg       0.53      0.54      0.53       600



## 6. Serialize Model and Encoder

Notice we need the LabelEncoder saved alongside the `.pkl` models directory. The API output must have a text response layer decoding arrays (`[0, 1, 2]`) back into `[High, Low, Medium]` natively down the road.

In [6]:
model_path   = WEIGHTS_DIR / "m2_rural_risk_model.pkl"
encoder_path = WEIGHTS_DIR / "m2_risk_encoder.pkl"

joblib.dump(model, model_path)
print(f"📦 Model base saved to       : {model_path}")

joblib.dump(le, encoder_path)
print(f"📦 Label Encoder saved to    : {encoder_path}")

📦 Model base saved to       : C:\Users\justi\Downloads\ML project data\Terraflux\models\weights\m2_rural_risk_model.pkl
📦 Label Encoder saved to    : C:\Users\justi\Downloads\ML project data\Terraflux\models\weights\m2_risk_encoder.pkl
